# 01. 데이터 모델링 (Type Alias · Interface · Optional 속성)

쇼핑몰의 **회원(User)** 과 **상품(Product)** 데이터를 타입으로 설계하고,
그 타입을 사용하는 함수 두 개(`filterUsersByRole`, `calculateTotalPrice`)를 구현합니다.

- 사용 개념: 리터럴 유니온 타입(`type`), `interface`, 선택 속성(`?`), 배열 타입(`User[]`), 매개변수/반환 타입
- 규칙: `any`는 한 번도 사용하지 않습니다.

## 1단계. 타입 설계 + 더미 데이터

1. `UserRole`, `Category`는 **리터럴 유니온 타입**으로 만들어 정해진 문자열만 들어오게 합니다.
   (예: `role: "manager"`라고 쓰면 컴파일 에러)
2. `User`, `Product`는 **interface**로 객체의 모양을 정의합니다.
3. `Product.discountRate`는 `?`를 붙여 **선택 속성**으로 만듭니다. 할인 없는 상품은 이 속성을 생략할 수 있고,
   이때 값의 타입은 `number | undefined`가 됩니다.
4. `users: User[]`, `products: Product[]`로 더미 데이터를 만들고 `console.log`로 출력해 정상 컴파일을 확인합니다.

In [ ]:
// 회원 역할은 세 가지 문자열 중 하나만 허용하는 리터럴 유니온 타입
type UserRole = "admin" | "customer" | "guest";

// 회원(User) 데이터의 모양을 정의하는 인터페이스
interface User {
  id: number;         // 회원 고유 번호
  name: string;       // 회원 이름
  email: string;      // 회원 이메일 (Product에는 없는 속성 → 타입 가드에 활용)
  role: UserRole;     // 회원 역할 (위에서 만든 UserRole만 가능)
  isActive: boolean;  // 활성 회원 여부
}

// 상품 카테고리도 정해진 값만 허용하는 리터럴 유니온 타입
type Category = "electronics" | "clothing" | "books";

// 상품(Product) 데이터의 모양을 정의하는 인터페이스
interface Product {
  id: number;            // 상품 고유 번호
  title: string;         // 상품명
  price: number;         // 정가 (원)
  category: Category;    // 상품 카테고리 (Category만 가능)
  stock: number;         // 재고 수량
  discountRate?: number; // 할인율 (0.1 = 10%), 선택 속성이라 없을 수도 있음 → number | undefined
}

// 회원 더미 데이터: User[] 타입을 명시해서 모든 원소가 User 모양인지 컴파일러가 검사
const users: User[] = [
  { id: 1, name: "김관리", email: "admin@shop.com", role: "admin", isActive: true },     // 관리자
  { id: 2, name: "이고객", email: "lee@shop.com", role: "customer", isActive: true },    // 일반 고객
  { id: 3, name: "박고객", email: "park@shop.com", role: "customer", isActive: false },  // 비활성 고객
  { id: 4, name: "최손님", email: "guest@shop.com", role: "guest", isActive: true },     // 비회원(게스트)
];

// 상품 더미 데이터: discountRate가 있는 상품과 없는 상품을 섞어서 준비
const products: Product[] = [
  { id: 101, title: "무선 키보드", price: 50000, category: "electronics", stock: 10, discountRate: 0.1 }, // 10% 할인
  { id: 102, title: "후드 티셔츠", price: 30000, category: "clothing", stock: 5 },                       // 할인 없음
  { id: 103, title: "타입스크립트 입문서", price: 20000, category: "books", stock: 3, discountRate: 0.2 }, // 20% 할인
];

// 더미 데이터가 타입 검사를 통과해 정상적으로 만들어졌는지 출력으로 확인
console.log("회원 목록:", users);
console.log("상품 목록:", products);

회원 목록: [
  {
    id: 1,
    name: '김관리',
    email: 'admin@shop.com',
    role: 'admin',
    isActive: true
  },
  {
    id: 2,
    name: '이고객',
    email: 'lee@shop.com',
    role: 'customer',
    isActive: true
  },
  {
    id: 3,
    name: '박고객',
    email: 'park@shop.com',
    role: 'customer',
    isActive: false
  },
  {
    id: 4,
    name: '최손님',
    email: 'guest@shop.com',
    role: 'guest',
    isActive: true
  }
]


상품 목록: [
  {
    id: 101,
    title: '무선 키보드',
    price: 50000,
    category: 'electronics',
    stock: 10,
    discountRate: 0.1
  },
  {
    id: 102,
    title: '후드 티셔츠',
    price: 30000,
    category: 'clothing',
    stock: 5
  },
  {
    id: 103,
    title: '타입스크립트 입문서',
    price: 20000,
    category: 'books',
    stock: 3,
    discountRate: 0.2
  }
]


## 2단계. `filterUsersByRole` — 역할별 회원 필터링

- 매개변수 `targetRole`의 타입을 `string`이 아닌 `UserRole`로 지정해서, 없는 역할(예: `"manager"`)을 넘기면 컴파일 단계에서 막힙니다.
- 반환 타입 `User[]`를 명시해 함수가 항상 회원 배열을 돌려준다는 것을 보장합니다.
- 내부는 배열의 고차 함수 `filter`를 사용합니다.

In [ ]:
// 회원 배열과 찾고 싶은 역할을 받아서, 그 역할을 가진 회원만 새 배열로 반환하는 함수
function filterUsersByRole(users: User[], targetRole: UserRole): User[] {
  // filter는 콜백이 true를 반환한 원소만 모아 새 배열을 만든다 (원본 배열은 변경되지 않음)
  return users.filter((user) => user.role === targetRole); // user는 User 타입으로 자동 추론됨
}

// "customer" 역할 회원만 조회
const customers: User[] = filterUsersByRole(users, "customer");
// 결과 확인: 이름만 뽑아서 보기 좋게 출력
console.log("customer 회원:", customers.map((user) => user.name));

// "admin" 역할 회원만 조회
const admins: User[] = filterUsersByRole(users, "admin");
console.log("admin 회원:", admins.map((user) => user.name));

customer 회원: [ '이고객', '박고객' ]


admin 회원: [ '김관리' ]


## 3단계. `calculateTotalPrice` — 할인율을 반영한 총 금액 계산

- 각 상품 1개씩 구매한다고 가정하고, **할인이 적용된 가격의 합계**를 구합니다.
- `discountRate`는 선택 속성이라 `undefined`일 수 있습니다. 그대로 `price * (1 - product.discountRate)`라고 쓰면
  strict 모드에서 `'product.discountRate' is possibly 'undefined'` 에러가 납니다.
- 그래서 `!== undefined` 비교로 **타입 좁히기(narrowing)** 를 한 뒤에만 할인율을 사용하고,
  없으면 할인율 0을 사용합니다. (같은 의미를 `product.discountRate ?? 0`으로 짧게 쓸 수도 있습니다.)
- 원 단위 소수점이 생기지 않도록 `Math.round`로 반올림합니다.

In [ ]:
// 상품 배열을 받아서 할인율을 반영한 총 금액(원)을 반환하는 함수
function calculateTotalPrice(products: Product[]): number {
  let total: number = 0; // 누적 합계를 담을 변수

  // 상품을 하나씩 꺼내서 가격을 더한다
  for (const product of products) {
    let rate: number = 0; // 기본 할인율은 0 (할인 없음)

    // discountRate는 number | undefined 이므로, undefined가 아닐 때만 사용 (타입 좁히기)
    if (product.discountRate !== undefined) {
      rate = product.discountRate; // 이 블록 안에서는 discountRate가 number로 좁혀짐
    }

    // 할인 적용 가격 = 정가 × (1 - 할인율)
    total += product.price * (1 - rate);
  }

  // 부동소수점 오차(예: 44999.99...)를 없애기 위해 원 단위로 반올림해서 반환
  return Math.round(total);
}

// 전체 상품의 할인 적용 총액 계산
const totalPrice: number = calculateTotalPrice(products);
// 50000×0.9 + 30000 + 20000×0.8 = 45000 + 30000 + 16000 = 91000
console.log(`할인 적용 총 금액: ${totalPrice}원`);

// 할인 상품이 하나도 없는 경우도 정상 동작하는지 확인 (후드 티셔츠만)
const noDiscountTotal: number = calculateTotalPrice(products.filter((product) => product.discountRate === undefined));
console.log(`할인 없는 상품만 계산: ${noDiscountTotal}원`);

할인 적용 총 금액: 91000원


할인 없는 상품만 계산: 30000원


## (참고) 타입 에러가 나는 경우

아래 코드는 **일부러 주석 처리**해 두었습니다. 주석을 풀면 컴파일 에러가 발생하는 예시입니다.

- `filterUsersByRole(users, "manager")` → `"manager"`는 `UserRole`에 없는 값이라 에러
- `{ ..., category: "food" }` → `"food"`는 `Category`에 없는 값이라 에러
- `product.price * (1 - product.discountRate)` → `discountRate`가 `undefined`일 수 있어서 에러 (strict 모드)

In [ ]:
// [에러 예시 1] UserRole에 없는 역할 → Argument of type '"manager"' is not assignable to parameter of type 'UserRole'.
// filterUsersByRole(users, "manager");

// [에러 예시 2] Category에 없는 카테고리 → Type '"food"' is not assignable to type 'Category'.
// const wrongProduct: Product = { id: 999, title: "라면", price: 1000, category: "food", stock: 1 };

// [에러 예시 3] 선택 속성을 좁히지 않고 바로 계산 → 'product.discountRate' is possibly 'undefined'.
// const unsafe = (product: Product): number => product.price * (1 - product.discountRate);

console.log("에러 예시는 모두 주석 처리되어 있어 이 셀은 정상 실행됩니다.");

에러 예시는 모두 주석 처리되어 있어 이 셀은 정상 실행됩니다.
